# Traitement TRI EER Agence — fusion recto/verso

Pipeline :
1. On repère les fichiers `id_recto_xxx.*` / `id_verso_xxx.*`
2. Conversion des images (jpeg/png) en PDF
3. Décision :
   - recto déjà à 2 pages (ou plus) → on garde tel quel, verso ignoré
   - recto à 1 page + verso présent → concaténation
   - recto à 1 page sans verso → on garde tel quel (anomalie loguée)
   - verso sans recto → anomalie loguée, on garde le verso seul
4. Passe de contrôle : sur les documents à 2 pages, on hash chaque page (phash) pour détecter
   les cas où recto+verso étaient en fait sur la même page scannée deux fois → on supprime le doublon
5. Sauvegarde intermédiaire (fichiers nommés par id, dans `_wip/`) — **pas encore de numérotation finale**,
   pour pouvoir faire les corrections manuelles ci-dessous sans décaler les numéros
6. Corrections manuelles (après vérif) : inversion de pages, exclusion de docs sans vrai verso
7. Numérotation finale `TRI_CNI_ETR_0001.pdf`, ... **sans trou** : uniquement sur les docs non exclus,
   les docs exclus restent tracés dans le CSV (statut + mention "supprimé")

Dépendances : `pip install pymupdf imagehash pillow pandas`


In [ ]:
import re, io, logging
from pathlib import Path
from collections import defaultdict

import fitz  # PyMuPDF
import imagehash
from PIL import Image
import pandas as pd

# --- config à adapter ---
INPUT_DIR = Path("classifier_eer/TRI/OK EER Agence")
OUTPUT_DIR = Path("output_tri")
WIP_DIR = OUTPUT_DIR / "_wip"                 # fichiers intermediaires, nommes par id
REJECTS_DIR = OUTPUT_DIR / "exclus_sans_verso"  # docs exclus manuellement
OUTPUT_DIR.mkdir(exist_ok=True)
WIP_DIR.mkdir(exist_ok=True)
REJECTS_DIR.mkdir(exist_ok=True)
PREFIX = "TRI_CNI_ETR"
HASH_THRESHOLD = 5  # distance de Hamming max pour considerer 2 pages identiques

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(OUTPUT_DIR / "traitement.log", mode="w", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger("tri_eer")


## 1. Repérage et regroupement des fichiers par id

In [ ]:
FILENAME_RE = re.compile(r"^(?P<id>.+?)_(?P<side>recto|verso)_[^.]+\.(?P<ext>pdf|jpe?g|png)$", re.IGNORECASE)

groups = defaultdict(dict)  # id -> {"recto": path, "verso": path}

for f in sorted(INPUT_DIR.iterdir()):
    if not f.is_file():
        continue
    m = FILENAME_RE.match(f.name)
    if not m:
        log.warning(f"Fichier ignore (pattern non reconnu): {f.name}")
        continue
    doc_id, side = m.group("id"), m.group("side").lower()
    if side in groups[doc_id]:
        log.warning(f"{doc_id}: plusieurs fichiers '{side}' trouves, {groups[doc_id][side].name} garde, {f.name} ignore")
        continue
    groups[doc_id][side] = f

log.info(f"{len(groups)} documents identifies")


## 2. Fonctions utilitaires : conversion en PDF + hash de page

In [ ]:
def to_pdf_doc(path: Path) -> fitz.Document:
    """Ouvre un fichier (pdf ou image) et renvoie un fitz.Document en pdf."""
    if path.suffix.lower() == ".pdf":
        return fitz.open(path)
    img_doc = fitz.open(path)
    pdf_bytes = img_doc.convert_to_pdf()
    img_doc.close()
    return fitz.open("pdf", pdf_bytes)


def page_hash(doc: fitz.Document, page_index: int) -> imagehash.ImageHash:
    pix = doc[page_index].get_pixmap(dpi=100)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    return imagehash.phash(img)


## 3. Décision recto/verso + concaténation

In [ ]:
records = []  # recap, un doc fitz "vivant" par entree pour l'instant

for doc_id, sides in groups.items():
    recto_path = sides.get("recto")
    verso_path = sides.get("verso")

    if recto_path is None and verso_path is None:
        continue

    if recto_path is None:
        merged = to_pdf_doc(verso_path)
        status = "verso seul (recto manquant)"
        log.warning(f"{doc_id}: {status}")
        records.append({"id": doc_id, "doc": merged, "status": status})
        continue

    recto_doc = to_pdf_doc(recto_path)

    if recto_doc.page_count >= 2:
        merged = recto_doc
        status = f"recto deja {recto_doc.page_count} pages -> verso non ajoute"
        if verso_path is not None:
            status += " (verso present mais ignore)"
        log.info(f"{doc_id}: {status}")
    elif verso_path is not None:
        verso_doc = to_pdf_doc(verso_path)
        merged = fitz.open()
        merged.insert_pdf(recto_doc)
        merged.insert_pdf(verso_doc)
        verso_doc.close()
        recto_doc.close()
        status = "recto (1 page) + verso concatenes"
        log.info(f"{doc_id}: {status}")
    else:
        merged = recto_doc
        status = "recto seul (verso manquant)"
        log.warning(f"{doc_id}: {status}")

    records.append({"id": doc_id, "doc": merged, "status": status})


## 4. Passe de contrôle par hashing

Sur les documents à 2 pages, on vérifie si les deux pages sont en fait identiques
(cas où recto+verso etaient deja sur le meme scan par ex.). Si la distance de Hamming
entre les deux phash est faible, on supprime la 2e page.

In [ ]:
for rec in records:
    doc = rec["doc"]
    if doc.page_count != 2:
        continue
    h0, h1 = page_hash(doc, 0), page_hash(doc, 1)
    dist = h0 - h1
    if dist <= HASH_THRESHOLD:
        doc.delete_page(1)
        rec["status"] += f" | pages identiques detectees (hamming={dist}) -> page 2 supprimee"
        log.info(f"{rec['id']}: pages 1 et 2 quasi-identiques (distance={dist}), page 2 supprimee")
    else:
        rec["status"] += f" | pages differentes (hamming={dist}), conservees"


## 5. Sauvegarde intermédiaire (par id, pas encore numérotée)

On écrit chaque doc dans `_wip/{id}.pdf` et on construit le `df`. La numérotation finale
`TRI_CNI_ETR_0001.pdf` se fera à l'étape 7, **après** les corrections manuelles, pour ne pas
laisser de trou de numérotation à cause d'une exclusion.

In [ ]:
for rec in records:
    wip_path = WIP_DIR / f"{rec['id']}.pdf"
    rec["doc"].save(wip_path)
    rec["doc"].close()
    rec["file"] = wip_path
    rec["exclu"] = False
    del rec["doc"]

df = pd.DataFrame(records)
df


## 6. Corrections manuelles post-vérification

Deux cas à traiter à la main après relecture du récap :
- le verso a été mis en page 1 par erreur de nommage → `inverser_pages`
- le "verso" contenait en fait le recto en double (donc pas de vrai verso dispo) → `exclure_sans_verso`

Les deux fonctions travaillent sur les fichiers `_wip/{id}.pdf` (donc rejouables, pas d'impact
sur la numérotation finale) et mettent à jour `df["status"]`.

In [ ]:
def inverser_pages(ids_a_inverser):
    """A utiliser apres verif manuelle : le verso a ete place en page 1 par erreur
    de nomenclature. Inverse les 2 pages et met a jour le statut dans df."""
    for doc_id in ids_a_inverser:
        mask = df["id"] == doc_id
        if not mask.any() or df.loc[mask, "exclu"].iloc[0]:
            log.warning(f"{doc_id}: id introuvable ou deja exclu, ignore")
            continue
        path = df.loc[mask, "file"].iloc[0]
        doc = fitz.open(path)
        if doc.page_count != 2:
            log.warning(f"{doc_id}: {doc.page_count} page(s), inversion ignoree (attendu 2 pages)")
            doc.close()
            continue
        doc.move_page(1, 0)
        tmp_path = path.with_suffix(".tmp.pdf")
        doc.save(tmp_path)
        doc.close()
        tmp_path.replace(path)
        df.loc[mask, "status"] = df.loc[mask, "status"] + " | pages inversees manuellement (verso etait en page 1)"
        log.info(f"{doc_id}: pages inversees")


def exclure_sans_verso(ids_a_exclure):
    """A utiliser apres verif manuelle : le "verso" contenait en fait le recto en double,
    donc pas de vrai verso disponible pour ce document. Le fichier est deplace dans
    'exclus_sans_verso' et le doc est marque exclu (il n'aura pas de numero final,
    mais reste trace dans le CSV)."""
    for doc_id in ids_a_exclure:
        mask = df["id"] == doc_id
        if not mask.any():
            log.warning(f"{doc_id}: id introuvable dans le recap, ignore")
            continue
        path = df.loc[mask, "file"].iloc[0]
        if path.exists():
            path.replace(REJECTS_DIR / path.name)
        df.loc[mask, "exclu"] = True
        df.loc[mask, "status"] = df.loc[mask, "status"] + " | document supprime : pas de verso reel (recto duplique), a rescanner"
        log.info(f"{doc_id}: document exclu")


In [ ]:
# a completer apres verification manuelle
ids_a_inverser = []  # ex: ["1bbznquy", "1fk5jx91"]
inverser_pages(ids_a_inverser)

ids_sans_verso_reel = []  # ex: ["1gto8s99"]
exclure_sans_verso(ids_sans_verso_reel)

df


## 7. Numérotation finale (sans trou)

À exécuter une fois les corrections manuelles terminées. Seuls les docs non exclus reçoivent
un numéro `TRI_CNI_ETR_0001.pdf`, `0002`, ... en séquence continue. Les docs exclus gardent
leur statut dans le CSV avec `output_file` vide, pour garder la trace.

In [ ]:
i = 0
for idx, rec in df.iterrows():
    if rec["exclu"]:
        df.loc[idx, "output_file"] = ""
        continue
    i += 1
    out_name = f"{PREFIX}_{i:04d}.pdf"
    rec["file"].replace(OUTPUT_DIR / out_name)
    df.loc[idx, "output_file"] = out_name
    log.info(f"{rec['id']} -> {out_name} ({rec['status']})")

df_final = df.drop(columns=["file"])
df_final.to_csv(OUTPUT_DIR / "recap_traitement.csv", index=False)
df_final
